In [2]:
!pip install -q datasets groq

from datasets import load_dataset
from groq import Groq
from google.colab import userdata
import pandas as pd
import re

dataset = load_dataset("nvidia/Nemotron-Personas-Korea", split="train")
df = dataset.to_pandas()

ojunseo = df[df['uuid'] == '1c48b1e108f04df38ad54716bd7eea07'].iloc[0]
staff = df[df['occupation'].str.contains('판매|영업|매장|서비스|안내', na=False) & (df['age'].between(23, 40))].sample(1, random_state=1).iloc[0]
traveler = df[df['hobbies_and_interests'].str.contains('여행', na=False) & (df['age'].between(25, 55)) & (df['uuid'] != '1c48b1e108f04df38ad54716bd7eea07')].sample(1, random_state=1).iloc[0]

print("데이터셋:", df.shape)
print("직원1:", staff['persona'][:40])
print("뒷손님:", traveler['persona'][:40])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.2 MB/s eta 0:00:00


README.md:   0%|          | 0.00/36.0k [00:00<?, ?B/s]

data/train-00000-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  220MB            

data/train-00000-of-00009.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  220MB            

...

data/train-00007-of-00009.parquet: downloading bytes:           |  0.00B            

data/train-00008-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  220MB            

data/train-00008-of-00009.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

데이터셋: (1000000, 26)
직원1: 박현주 씨는 이른 사회생활로 광명시에 내 집 마련을 이룬 실속 있는 성격
뒷손님: 이은애 씨는 김포에서 아이와 어머니를 돌보며 꼼꼼한 자산 관리와 건강 관


In [3]:
characters = {
    "오준서": {"row": ojunseo, "role": "인터넷면세점으로 구매한 면세품을 찾으러 온 손님. 일본 여행을 앞두고 있음. 지금 데스크 앞에서 본인 차례를 맞이함."},
    "박현주": {"row": staff, "role": "롯데면세점 인터넷면세점 픽업 데스크 직원. 오준서를 응대하고 있음."},
    "이은애": {"row": traveler, "role": "오준서 바로 뒷순번으로, 자신도 인터넷면세점 상품을 찾으러 대기 중인 손님."}
}
situation = "롯데면세점 인터넷면세점 상품 수령 데스크. 오준서는 일본 여행을 앞두고 인터넷으로 구매한 면세품을 찾으러 왔다. 박현주는 응대 직원이다. 이은애는 오준서 뒷순번으로 대기 중이다."
order = ["오준서", "박현주", "이은애"]

def build_system_prompt(name, info):
    row = info["row"]
    return f"""당신은 '{name}'이라는 인물입니다. 아래는 당신의 페르소나입니다.

[기본 정보] 나이: {row['age']}세, 성별: {row['sex']}, 직업: {row['occupation']}, 거주지: {row['district']}
[페르소나] {row['persona']}
[전문성] {row.get('professional_persona', '')}
[취미와 관심사] {row.get('hobbies_and_interests', '')}
[가치관/배경] {row.get('cultural_background', '')}

[상황] 롯데면세점 인터넷면세점 상품 수령 데스크입니다.
당신의 역할: {info['role']}

지시사항:
- 위 페르소나에 담긴 당신의 성격, 취미, 가치관, 말투가 대사에 자연스럽게 드러나도록 연기하세요.
- 반드시 순수 한국어로만 답변하세요. 한자, 일본어, 영어 단어를 절대 섞지 마세요. 사람 이름도 한글로 부르세요.
- 한두 문장 구어체로 말하고, 필요하면 [행동: OOO] 형식으로 표시하세요."""

def contains_foreign_chars(text):
    allowed = re.compile(r'^[\uAC00-\uD7A3\u3131-\u318E\s0-9.,!?~()\[\]:;\'\"\-…%·/&+*⚠]*$')
    return not bool(allowed.match(text))

print("설정 완료")

설정 완료


In [4]:
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def get_reply_groq(name, info, situation, log, max_retries=5):
    sp = build_system_prompt(name, info)
    log_text = "\n".join(log) if log else "(아직 대화 없음, 상황이 막 시작됨)"
    up = f"""[상황] {situation}

[지금까지의 대화]
{log_text}

이제 당신의 차례입니다. 위 흐름에 이어서 자연스럽게 한두 문장으로 대사와 필요하면 행동을 말하세요.

주의사항:
- 반드시 순수 한국어만 사용하세요.
- 상대를 '선생님'이라고 부르지 마세요. 손님은 직원을 '저기요' 또는 호칭 없이 부르고, 직원은 손님을 '고객님' 또는 이름+'님'으로 부르세요.
- 당신의 이름과 상대의 이름을 정확히 쓰세요. 오타를 내지 마세요.
- 당신의 역할(손님인지 직원인지)에 맞게 말하세요."""
    for _ in range(max_retries):
        r = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role":"system","content":sp},{"role":"user","content":up}],
            temperature=0.7, max_tokens=200
        )
        reply = r.choices[0].message.content.strip()
        if not contains_foreign_chars(reply):
            return reply
    return re.sub(r'[^\uAC00-\uD7A3\u3131-\u318E\s0-9.,!?~()\[\]:;\'\"\-…%·/&+*]', '', reply)

conversation_log_groq = []
for turn in range(3):
    print(f"\n{'='*50}\n[턴 {turn+1}] (상용 API - Llama 70B)\n{'='*50}")
    for name in order:
        reply = get_reply_groq(name, characters[name], situation, conversation_log_groq)
        conversation_log_groq.append(f"{name}: {reply}")
        print(f"\n[{name}]\n{reply}")


[턴 1] (상용 API - Llama 70B)

[오준서]
박현주, 안녕하세요. 오준서입니다. 저기요, 제가 인터넷으로 구매한 면세품 수령하러 왔는데요, 확인해 주실 수 있을까요? [행동: 구매내역 확인하는 화면을 보여주는 스마트폰을 꺼냄]

[박현주]
오준서님, 안녕하세요. 저는 박현주입니다. 잠시만 기다려주세요, 주문번호와 성함으로 확인해 보겠습니다. [행동: 컴퓨터 화면을 확인하며 키보드를 타이핑함]

[이은애]
박현주, 안녕하세요. 저는 이은애입니다. 혹시 제가 찾으러 온 상품도 확인해 줄 수 있을까요? [행동: 지갑에서 주문 확인서를 꺼냄]

[턴 2] (상용 API - Llama 70B)

[오준서]
박현주, 죄송한데요, 지금 제 차례 아닌가요? [행동: 조금 앞으로 걸어가며 질문함]

[박현주]
오준서님, 잠시만 기다려주세요. 저는 지금 오준서님의 주문번호를 확인 중이니까요. [행동: 컴퓨터 화면을 다시 확인하며 키보드를 타이핑함] 이은애님, 먼저 오준서님께 응대하겠습니다. 잠시만 기다려 주세요.

[이은애]
박현주, 죄송해요. 주문번호를 확인해 주셔야 제가 상품을 수령할 수 있잖아요. [행동: 주문 확인서를 더 가까이 보여줌]

[턴 3] (상용 API - Llama 70B)

[오준서]
박현주, 죄송한데요, 제 차례인데요, 제 주문번호를 확인해 주시면 안 될까요? [행동: 조금 더 앞으로 나서며 스마트폰을 다시 보여줌]

[박현주]
오준서님, 잠시만 더 기다려 주세요. 제가 지금 주문번호를 확인하고 있습니다. [행동: 컴퓨터 화면을 다시 확인하며 키보드를 타이핑함] 이은애님, 잠시만 더 기달려 주세요. 순서대로 처리하겠습니다.

[이은애]
박현주, 오준서님이 주문번호 확인을 마치셨으니 이제 제가 상품을 찾으러 왔어요. 주문 확인서를 보여드릴게요. [행동: 주문 확인서를 더 가까이 보여줌]


In [5]:
with open("result_groq.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(conversation_log_groq))
print("상용 API 결과 백업 완료")

상용 API 결과 백업 완료


In [6]:
import gc
try:
    del df
    gc.collect()
    print("df 메모리 해제 완료")
except:
    print("df 이미 없음")

!pip install -q transformers accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
local_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("모델 로드 완료:", next(local_model.parameters()).device)

df 메모리 해제 완료


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

모델 로드 완료: cuda:0


In [7]:
def get_reply_local(name, info, situation, log, max_retries=4):
    sp = build_system_prompt(name, info)
    log_text = "\n".join(log) if log else "(아직 대화 없음, 상황이 막 시작됨)"
    up = f"""[상황] {situation}

[지금까지의 대화]
{log_text}

이제 당신의 차례입니다. 위 흐름에 이어서 자연스럽게 한두 문장으로 대사와 필요하면 행동을 말하세요.

주의사항:
- 반드시 순수 한국어만 사용하세요.
- 상대를 '선생님'이라고 부르지 마세요. 직원은 손님을 '고객님' 또는 이름+'님', 손님은 직원을 '저기요'로 부르세요.
- 당신의 역할(손님인지 직원인지)에 맞게 말하세요."""

    messages = [{"role": "system", "content": sp}, {"role": "user", "content": up}]

    for _ in range(max_retries):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(local_model.device)
        with torch.no_grad():
            outputs = local_model.generate(
                **inputs, max_new_tokens=150, temperature=0.7,
                do_sample=True, pad_token_id=tokenizer.eos_token_id
            )
        reply = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        if not contains_foreign_chars(reply):
            return reply
    return re.sub(r'[^\uAC00-\uD7A3\u3131-\u318E\s0-9.,!?~()\[\]:;\'\"\-…%·/&+*]', '', reply)

conversation_log_local = []
for turn in range(3):
    print(f"\n{'='*50}\n[턴 {turn+1}] (로컬 3B - Qwen2.5)\n{'='*50}")
    for name in order:
        reply = get_reply_local(name, characters[name], situation, conversation_log_local)
        conversation_log_local.append(f"{name}: {reply}")
        print(f"\n[{name}]\n{reply}")


[턴 1] (로컬 3B - Qwen2.5)

[오준서]
안녕하세요, 저기요. 일본 여행 준비를 위해 인터넷으로 주문한 면세품을 찾으러 왔어요. 선생님, 혹시 저의 차례가 왔나요? [행동: 고개를 젓고, 손님으로부터 조금 떨어져 서서 기다림]

[박현주]
저기요, 안녕하세요. 일본 여행 준비를 위해 인터넷으로 주문한 제품을 확인하러 왔습니다. 고객님, 혹시 제가 차례가 왔나요? [행동: 손님에게 다가가서 서서 기다림]

[이은애]
이은애: 안녕하세요, 저기요. 일본 여행 준비를 위해 인터넷으로 주문한 제품을 확인하러 왔어요. 고객님, 혹시 저의 차례가 왔나요? [행동: 오준서 뒤로 물러서서 손님으로부터 조금 떨어진 곳에서 기다림]

[턴 2] (로컬 3B - Qwen2.5)

[오준서]
오준서: 안녕하세요, 고객님. 혹시 저의 차례가 왔나요? [행동: 손님들의 차례표를 살펴보면서 서 있는 곳에서 고객님들에게 다가감]

[박현주]
박현주: 고객님, 혹시 저의 차례가 왔나요? [행동: 손님들이 차례표를 살펴보는 모습을 보며, 손님들의 차례가 왔음을 알리는 데가 아닌, 손님이 먼저 차례표를 확인하도록 돕기 위해 서 있는 곳으로 다가갑니다.]

[이은애]
이은애: 고객님, 혹시 저의 차례가 왔나요? [행동: 오준서에게 다가가서 서 있던 위치에서 조금 떨어진 곳으로 이동하여 고객님들과 가까워짐]

[턴 3] (로컬 3B - Qwen2.5)

[오준서]
오준서: 고객님, 혹시 저의 차례가 왔나요? [행동: 손님들 차례표를 살펴보며 서 있는 곳으로 다가감]

[박현주]
박현주: 고객님, 혹시 저의 차례가 왔나요? [행동: 차례표를 확인하는 손님들의 모습을 살피며, 차례가 왔음을 확인하고 서 있는 곳으로 다가갑니다.]

[이은애]
이은애: 고객님, 혹시 저의 차례가 왔나요? [행동: 오준서와 박현주에게 다가가서 서 있던 위치에서 조금 떨어진 곳으로 이동하여 손님들의 차례표를 살펴보며 고객님들과 가까워짐]


In [8]:

with open("result_local.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(conversation_log_local))
print("로컬 3B 결과 백업 완료")

로컬 3B 결과 백업 완료


In [9]:
print("=== 상용 API 결과 (개수) ===", len(conversation_log_groq))
print(conversation_log_groq[0][:60])  # 첫 대사 미리보기
print()
print("=== 로컬 3B 결과 (개수) ===", len(conversation_log_local))
print(conversation_log_local[0][:60])  # 첫 대사 미리보기

=== 상용 API 결과 (개수) === 9
오준서: 박현주, 안녕하세요. 오준서입니다. 저기요, 제가 인터넷으로 구매한 면세품 수령하러 왔는데요, 확인

=== 로컬 3B 결과 (개수) === 9
오준서: 안녕하세요, 저기요. 일본 여행 준비를 위해 인터넷으로 주문한 면세품을 찾으러 왔어요. 선생님, 혹
